# Modeling Objectives and Questions

In this modeling stage, we focus on **two main outcomes**:

1. **Prediction**: Can the four structural properties predict CO2 uptake for a MOF at each pressure?
2. **Scientific understanding**: Which structural properties influence CO2 uptake most, and does their influence change with pressure?

## Core Modeling Question

> Using LCD, PLD, void fraction, and surface area, how accurately can we predict CO2 uptake at different pressures, and which features are most important at each pressure?

## What We Will Examine

- Whether model performance differs between **low-pressure** and **high-pressure** regimes.
- Whether **surface area** or **void fraction** becomes more influential at higher pressure.
- Whether **LCD** and **PLD** provide distinct predictive value despite their strong correlation.
- Whether **nonlinear models** outperform **linear models**, consistent with nonlinear patterns observed in EDA.

## Features Used

- **LCD** (Largest Cavity Diameter)
- **PLD** (Pore Limiting Diameter)
- **Void fraction**
- **Surface area**

## Target

- **CO2 uptake** at each pressure point
==============================================================

In [2]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Load Data
data = pd.read_csv('/home/susan/mof-co2-adsorption/data/processed/data_clean_v2')
# Copy dataframe
df= data.copy()
print(df.columns)
print("="*30)


Index(['filename', 'lcd', 'pld', 'void_fraction', 'surface_area_m2g', 'mofid',
       'CO2_uptake_0.01bar_molkg', 'CO2_uptake_0.05bar_molkg',
       'CO2_uptake_0.1bar_molkg', 'CO2_uptake_0.5bar_molkg',
       'CO2_uptake_2.5bar_molkg'],
      dtype='object')


In [20]:
# Step 2 Ml processing 
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, r2_score, mean_absolute_error
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder, PolynomialFeatures, StandardScaler, MinMaxScaler  # for l
from sklearn.utils import shuffle
from sklearn.metrics import roc_curve
from sklearn.metrics import roc_auc_score

In [15]:
# Store the four structural features used in the relationship analysis.
feature_columns = ["lcd", "pld", "void_fraction", "surface_area_m2g"]

# Store the five CO2-uptake targets in increasing pressure order.
target = "CO2_uptake_0.01bar_molkg"

# Select only the required features and targets
features = df[feature_columns].copy()
target = df["CO2_uptake_0.01bar_molkg"].copy()
# split data
# temp simply means temporary.
# X_train: 60%
# X_temp: 40%

X_train, X_temp, y_train, y_temp = train_test_split(features , target, test_size = 0.4 ,
                                                        random_state = 12345)
                                                        # stratify=target means:
                                                        # Keep the same target-class proportions in the training
                                                        #  and validation sets as in the original dataset.Use stratification when:
                                                        # You have a classification problem
                                                        # Your target contains classes, such as 0 and 1
                                                        # The classes are imbalanced
                                                        # You want every dataset to represent the original class distribution

# Train: 60%
# Validation: 20%
# Test: 20%
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=12345
)
print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_valid.shape, y_valid.shape)
print("Testing:", X_test.shape, y_test.shape)

print("\nMissing values in training features:")
print(X_train.isna().sum())


Training: (18740, 4) (18740,)
Validation: (6247, 4) (6247,)
Testing: (6247, 4) (6247,)

Missing values in training features:
lcd                 0
pld                 0
void_fraction       0
surface_area_m2g    0
dtype: int64


In [16]:

# Create simple baseline model
dummy_model = DummyRegressor(strategy="mean")
dummy_model.fit(X_train, y_train)

dummy_predictions = dummy_model.predict(X_valid)
print(dummy_predictions[:2])

[0.12550654 0.12550654]


==============================================================

The Dummy Regressor was used as a baseline. It predicts the mean CO₂ uptake at each pressure without using the structural features. Therefore, all later models should outperform it to demonstrate that they learn meaningful structure–property relationships.

==============================================================


 *Targets are continuous numerical values CO₂ uptake in mol/kg,so this is a regression problem.*

- Linear regression: predicts continuous values such as 2.4 mol/kg.
- Logistic regression: predicts categories such as high/low uptake or 0/1.

In [31]:
#Train  Linear Regression Model
# Create a linear regression model, train it on your training data, 
# and make predictions on your test data
# Your task: Create a train the model
model = LinearRegression()
# Train the model
model.fit(X_train, y_train)
# Make predictions
linear_predictions = model.predict(X_valid)

mse = mean_squared_error(y_valid, predictions)
rmse = np.sqrt(mse)

print(f"Mean Squared Error: {mse:.2f}")
print(f"Root Mean Squared Error: {rmse:.2f}")
print(linear_predictions[:5])

Mean Squared Error: 0.04
Root Mean Squared Error: 0.20
[-0.06636538  0.16500986  0.21746029  0.25112625  0.14861813]


In [32]:
#Train Ridge regression. Scaling is necessary for Ridge.
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])

ridge_model.fit(X_train, y_train)
ridge_predictions = ridge_model.predict(X_valid)
print(ridge_predictions[:5])

[-0.06631551  0.16500048  0.21743688  0.25113649  0.14860263]


In [33]:
comparison = pd.DataFrame({
    "Actual": y_valid.iloc[:5],
    "Predicted_ridge": ridge_predictions[:5],
    "Predicted_linear":linear_predictions[:5]

})

print(comparison)

         Actual  Predicted_ridge  Predicted_linear
21214  0.023901        -0.066316         -0.066365
25107  0.051266         0.165000          0.165010
2383   0.167260         0.217437          0.217460
15497  0.329274         0.251136          0.251126
29148  0.066501         0.148603          0.148618


### Linear vs. Ridge Regression (0.01 bar)

Ridge and Linear Regression produced nearly identical predictions for CO2 uptake at **0.01 bar**, suggesting that Ridge regularization had minimal impact in this setting.

**Key observations:**
- Predictions include both overestimation and underestimation.
- At least one prediction is negative, which is not physically meaningful for CO2 uptake.
- This indicates that simple linear models may not fully capture adsorption behavior at very low pressure.

**Implication for next step:**
- Evaluate nonlinear models to test whether they improve predictive accuracy and physical realism.



In [34]:

# Store the validation results for each model
results = []

# Evaluate the three models
models_predictions = {
    "Dummy": dummy_predictions,
    "Linear Regression": linear_predictions,
    "Ridge Regression": ridge_predictions
}

for model_name, predictions in models_predictions.items():

    mae = mean_absolute_error(y_valid, predictions)
    rmse = np.sqrt(mean_squared_error(y_valid, predictions))
    r2 = r2_score(y_valid, predictions)

    results.append({
        "Model": model_name,
        "MAE": mae,
        "RMSE": rmse,
        "R²": r2
    })

# Create the comparison table
comparison_table = pd.DataFrame(results)

comparison_table

,Model,MAE,RMSE,R²
0,Dummy,0.127524,0.224469,-0.000283
1,Linear Regression,0.110546,0.198692,0.216257
2,Ridge Regression,0.110542,0.198692,0.216256


### Validation Performance Summary (0.01 bar)

The Linear and Ridge models are better than the Dummy baseline, but their predictive performance is still limited.

- **Dummy:** MAE = 0.128, RMSE = 0.224, R2 approx. 0.00  
  It explains none of the variation in CO2 uptake.
- **Linear Regression:** MAE = 0.111, RMSE = 0.199, R2 = 0.216  
  It explains about **21.6%** of the variation at 0.01 bar.
- **Ridge Regression:** almost identical to Linear Regression.  
  Ridge regularization had little effect.

**Short interpretation:**

> Linear and Ridge Regression outperform the Dummy baseline, showing that the four structural features contain some useful information for predicting CO2 uptake at 0.01 bar. However, their R2 of approximately 0.216 means they explain only 21.6% of the variation, indicating weak predictive performance. Their nearly identical results show that Ridge regularization provides no meaningful improvement. This limited performance is reasonable at very low pressure, where CO2 uptake may depend strongly on chemical binding characteristics not included in the four structural features.

In [ ]:
# RandomForest model